# Tutorial 1 — Attention cho phân loại câu với IMDB

**Mục tiêu:** hiểu trực giác Attention; Q/K/V; Scaled Dot-Product Attention; xây dựng `Embedding → BiLSTM → Attention → Classifier`; huấn luyện trên IMDB; quan sát attention weights.

> Khuyến nghị Colab: `Runtime → Change runtime type → T4 GPU`.

Bài toán: review phim → `0 = Negative`, `1 = Positive`.

## 1. Tại sao cần Attention?

RNN/LSTM truyền thống thường nén câu vào một vector. Với câu dài, điều này tạo bottleneck và không phải token nào cũng quan trọng như nhau.

Attention học trọng số \(\alpha_t\) cho từng hidden state \(h_t\):

\[
\sum_t\alpha_t=1,\qquad c=\sum_t\alpha_t h_t
\]

`c` là context vector dùng để phân loại cả câu.

## 2. Query, Key, Value

Scaled Dot-Product Attention:

\[
\mathrm{Attention}(Q,K,V)=
\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
\]

- **Query:** đang tìm thông tin gì?
- **Key:** dùng để so khớp với query.
- **Value:** thông tin được tổng hợp.

Ba bước: similarity → softmax → weighted sum.

In [ ]:
import numpy as np

def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

Q = np.array([[1.0, 0.5]])
K = np.array([[1.,0.],[0.,1.],[1.,1.]])
V = np.array([[1.,0.],[0.,1.],[1.,1.]])

scores = Q @ K.T / np.sqrt(K.shape[-1])
weights = softmax(scores)
context = weights @ V

print("Scores:", scores)
print("Attention weights:", weights)
print("Context vector:", context)

## 3. Attention pooling cho sentence classification

BiLSTM sinh \(H=[h_1,\dots,h_T]\). Ta học:

\[
e_t=v^\top\tanh(Wh_t+b),\qquad
\alpha_t=\mathrm{softmax}(e_t),\qquad
c=\sum_t\alpha_t h_t
\]

Pipeline:

```text
Tokens → Embedding → BiLSTM → Attention → Context → Dense → Sentiment
```

In [ ]:
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 4. Load IMDB

Keras cung cấp sẵn 25k train + 25k test. `FAST_MODE=True` giúp lab chạy nhanh.

In [ ]:
VOCAB_SIZE = 20_000
MAX_LEN = 250
FAST_MODE = True

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE
)

if FAST_MODE:
    x_train, y_train = x_train[:12_000], y_train[:12_000]
    x_test, y_test = x_test[:5_000], y_test[:5_000]

print("Train:", len(x_train), " Test:", len(x_test))

In [ ]:
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {idx + 3: word for word, idx in word_index.items()}
reverse_word_index.update({0:"<PAD>",1:"<START>",2:"<UNK>",3:"<UNUSED>"})

def decode_review(seq):
    return " ".join(reverse_word_index.get(int(i), "?") for i in seq)

print(decode_review(x_train[0])[:1500])
print("\nLabel:", "Positive" if y_train[0] else "Negative")

## 5. Padding
Cắt chuỗi dài và thêm `<PAD>` cho chuỗi ngắn.

In [ ]:
x_train_pad = keras.utils.pad_sequences(
    x_train, maxlen=MAX_LEN, padding="post", truncating="post"
)
x_test_pad = keras.utils.pad_sequences(
    x_test, maxlen=MAX_LEN, padding="post", truncating="post"
)
print(x_train_pad.shape, x_test_pad.shape)

## 6. Baseline không dùng Attention

```text
Embedding → GlobalAveragePooling → Dense
```

Mục tiêu là có một mốc đơn giản để so sánh.

In [ ]:
baseline = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(VOCAB_SIZE, 64, mask_zero=True),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])
baseline.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
baseline.summary()

In [ ]:
baseline.fit(
    x_train_pad, y_train,
    validation_split=0.2,
    epochs=2 if FAST_MODE else 4,
    batch_size=128,
    verbose=1
)
_, baseline_acc = baseline.evaluate(x_test_pad, y_test, verbose=0)
print(f"Baseline test accuracy: {baseline_acc:.4f}")

## 7. Tự xây Attention Layer
Layer học score cho từng timestep và bỏ qua padding bằng mask.

In [ ]:
class AttentionPooling(layers.Layer):
    def __init__(self, attention_dim=64, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True
        self.proj = layers.Dense(attention_dim, activation="tanh")
        self.score = layers.Dense(1, use_bias=False)

    def call(self, inputs, mask=None):
        e = tf.squeeze(self.score(self.proj(inputs)), axis=-1)
        if mask is not None:
            e = tf.where(mask, e, tf.cast(-1e9, e.dtype))
        alpha = tf.nn.softmax(e, axis=1)
        context = tf.reduce_sum(
            inputs * tf.expand_dims(alpha, axis=-1), axis=1
        )
        return context, alpha

    def compute_mask(self, inputs, mask=None):
        return (None, None)

## 8. BiLSTM + Attention

`return_sequences=True` vì Attention cần hidden state ở từng vị trí.

In [ ]:
inputs = keras.Input(shape=(MAX_LEN,), dtype="int32")
x = layers.Embedding(VOCAB_SIZE, 96, mask_zero=True)(inputs)
h = layers.Bidirectional(
    layers.LSTM(64, return_sequences=True)
)(x)

att_layer = AttentionPooling(64, name="attention")
context, att_weights = att_layer(h)

x = layers.Dense(64, activation="relu")(context)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

attention_model = keras.Model(inputs, outputs)
attention_explainer = keras.Model(inputs, [outputs, att_weights])

attention_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
attention_model.summary()

In [ ]:
history = attention_model.fit(
    x_train_pad, y_train,
    validation_split=0.2,
    epochs=3 if FAST_MODE else 5,
    batch_size=128,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=1, restore_best_weights=True
        )
    ],
    verbose=1
)

## 9. Đánh giá trên test set

In [ ]:
_, test_acc = attention_model.evaluate(x_test_pad, y_test, verbose=0)
print(f"Baseline accuracy : {baseline_acc:.4f}")
print(f"Attention accuracy: {test_acc:.4f}")

plt.figure(figsize=(7,4))
plt.plot(history.history["loss"], marker="o", label="Train")
plt.plot(history.history["val_loss"], marker="o", label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.show()

## 10. Attention đang nhìn vào từ nào?

Ta xem các token có \(\alpha_t\) lớn nhất.

> Attention weight hữu ích để quan sát mô hình, nhưng **không nên xem là causal explanation tuyệt đối**.

In [ ]:
def explain_review(index=10, top_k=25):
    seq = x_test_pad[index:index+1]
    pred, weights = attention_explainer.predict(seq, verbose=0)
    prob = float(pred[0,0])
    weights = weights[0]

    rows = []
    for token_id, w in zip(seq[0], weights):
        if token_id != 0:
            rows.append((
                reverse_word_index.get(int(token_id), "?"),
                float(w)
            ))

    df_att = pd.DataFrame(rows, columns=["token","attention"])
    top = df_att.sort_values("attention", ascending=False).head(top_k)

    print("True:", "Positive" if y_test[index] else "Negative")
    print("Pred:", "Positive" if prob >= .5 else "Negative")
    print("P(positive):", round(prob,4))
    print("\nReview:\n", " ".join(df_att["token"].tolist())[:2200])
    display(top)

    plot_df = top.sort_values("attention")
    plt.figure(figsize=(8,7))
    plt.barh(plot_df["token"], plot_df["attention"])
    plt.xlabel("Attention weight")
    plt.show()

explain_review(10)

## 11. Dự đoán review tự viết

In [ ]:
def encode_text(text):
    seq = [1]
    for token in text.lower().split():
        idx = word_index.get(token)
        token_id = 2 if idx is None else idx + 3
        seq.append(token_id if token_id < VOCAB_SIZE else 2)

    return keras.utils.pad_sequences(
        [seq], maxlen=MAX_LEN, padding="post", truncating="post"
    )

def predict_sentiment(text):
    prob = float(attention_model.predict(
        encode_text(text), verbose=0
    )[0,0])
    label = "Positive" if prob >= .5 else "Negative"
    print(f"{label} — P(positive)={prob:.4f}")

predict_sentiment("this movie is wonderful touching and beautifully acted")
predict_sentiment("this movie is boring terrible and a complete waste of time")

# 12. Tổng kết và bài tập

Đã học: weighted aggregation; Q/K/V; Scaled Dot-Product Attention; BiLSTM + Attention; sentence classification; attention visualization.

### Bài tập
1. Thay BiLSTM bằng GRU.
2. So sánh `MAX_LEN=100, 250, 500`.
3. Phân tích 5 review dự đoán sai và các token có attention cao.
4. Tự cài đặt Scaled Dot-Product Self-Attention.
5. Giải thích vì sao phải chia \(QK^\top\) cho \(\sqrt{d_k}\).
6. So sánh single-head và multi-head attention.